# AgentRAG-Tutor — PPO Difficulty Trainer (Colab / Jupyter)

Trains the PPO policy described in **Paper Section 5.3** on the BKT-driven
tutoring simulator. Designed to run on a **free-tier GPU** (T4 / V100 / A100).
Outputs a checkpoint zip you can drop into the repo at
`backend/models/ppo_agent/final_model.zip`.

## Quick start

1. Open this notebook in **Google Colab** (`Runtime → Change runtime type → GPU`).
2. Run all cells top-to-bottom.
3. The final cell triggers a download of `final_model.zip` (~150 KB).
4. Replace `backend/models/ppo_agent/final_model.zip` in the repo with that file.
5. Restart the FastAPI backend — it will auto-load the new policy.

Total wall-clock on a Colab T4: **~5–8 minutes** for the paper-spec 500K
timesteps (compared to ~30–45 min on CPU).

---

## 1 · Install dependencies

On Colab `torch` is pre-installed; we add `stable-baselines3` and
`gymnasium`. Install is silent except for warnings.

In [ ]:
!pip install -q "stable-baselines3>=2.3" "gymnasium>=0.29" "tensorboard>=2.15"

In [ ]:
import torch, stable_baselines3 as sb3, gymnasium
print('torch  :', torch.__version__)
print('sb3    :', sb3.__version__)
print('gym    :', gymnasium.__version__)
print('cuda   :', torch.cuda.is_available())
print('device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2 · Knowledge Component config (Paper Section 5.2.2 — Table 2)

These five 4-parameter HMMs drive the simulator's student model. The PPO
agent learns to pick difficulty levels that maximise total mastery gain
across all five KCs.

In [ ]:
BKT_PARAMS = {
    1: {'name': 'Agents & ML Intro',        'p_l0': 0.35, 'p_t': 0.20, 'p_g': 0.25, 'p_s': 0.10},
    2: {'name': 'Search & CSP',             'p_l0': 0.20, 'p_t': 0.15, 'p_g': 0.20, 'p_s': 0.10},
    3: {'name': 'Knowledge Representation', 'p_l0': 0.25, 'p_t': 0.18, 'p_g': 0.22, 'p_s': 0.08},
    4: {'name': 'Planning & Heuristics',    'p_l0': 0.20, 'p_t': 0.12, 'p_g': 0.18, 'p_s': 0.10},
    5: {'name': 'Learning & Game AI',       'p_l0': 0.15, 'p_t': 0.10, 'p_g': 0.20, 'p_s': 0.12},
}
MASTERY_THRESHOLD = 0.95

## 3 · BKT model + 5-KC tracker

Self-contained copy of `backend/app/services/ai/bkt/bkt_tracker.py` so the
notebook runs without the rest of the repo.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class BKTModel:
    name: str
    p_l0: float; p_t: float; p_g: float; p_s: float
    p_l: float = 0.0
    history: List[float] = field(default_factory=list)

    def __post_init__(self):
        self.p_l = self.p_l0
        self.history = [self.p_l0]

    def update(self, correct: bool) -> float:
        if correct:
            num = self.p_l * (1 - self.p_s)
            den = num + (1 - self.p_l) * self.p_g
        else:
            num = self.p_l * self.p_s
            den = num + (1 - self.p_l) * (1 - self.p_g)
        post = num / max(den, 1e-9)
        self.p_l = post + (1 - post) * self.p_t
        self.history.append(self.p_l)
        return self.p_l

    def predict_correct(self) -> float:
        return self.p_l * (1 - self.p_s) + (1 - self.p_l) * self.p_g

    def is_mastered(self) -> bool:
        return self.p_l >= MASTERY_THRESHOLD


class BKTTracker:
    def __init__(self):
        self.models: Dict[int, BKTModel] = {
            kc: BKTModel(name=p['name'], p_l0=p['p_l0'], p_t=p['p_t'],
                         p_g=p['p_g'], p_s=p['p_s'])
            for kc, p in BKT_PARAMS.items()
        }
    def update(self, kc, correct):       return self.models[kc].update(correct)
    def get_mastery_vector(self):        return [self.models[k].p_l for k in sorted(self.models)]
    def get_weakest_kc(self):            return min(self.models, key=lambda k: self.models[k].p_l)
    def all_mastered(self):              return all(m.is_mastered() for m in self.models.values())
    def predict_next_correct(self, kc):  return self.models[kc].predict_correct() if kc in self.models else 0.5

## 4 · Custom Gymnasium environment (Paper Section 5.3.1 MDP)

* **State** `s_t = [P(L_1..5), session_step / max_steps]` — `Box(0, 1, shape=(6,))`
* **Action** `a_t ∈ {0..4}` → difficulty level `1..5`
* **Reward** `r_t = Σ_k ΔP(L_t^k) − λ·max(0, 3−a_t)` (penalises too-easy)

In [ ]:
import gymnasium as gym
import numpy as np

class TutoringEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, max_steps: int = 20, lambda_pen: float = 0.05):
        super().__init__()
        self.max_steps  = max_steps
        self.lambda_pen = lambda_pen
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(6,), dtype=np.float32)
        self.action_space      = gym.spaces.Discrete(5)
        self.tracker = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.tracker = BKTTracker()
        self.step_count = 0
        return self._obs(), {}

    def step(self, action):
        difficulty = int(action) + 1
        prev = sum(self.tracker.get_mastery_vector())
        kc   = self.tracker.get_weakest_kc()
        p_correct = self.tracker.predict_next_correct(kc)
        diff_factor = {1: 0.85, 2: 0.75, 3: 0.60, 4: 0.45, 5: 0.30}[difficulty]
        correct = np.random.random() < (p_correct * diff_factor)
        self.tracker.update(kc, correct)
        new = sum(self.tracker.get_mastery_vector())
        reward = (new - prev) - self.lambda_pen * max(0, 3 - difficulty)
        self.step_count += 1
        terminated = self.tracker.all_mastered() or self.step_count >= self.max_steps
        return self._obs(), reward, terminated, False, {
            'difficulty': difficulty, 'kc': kc, 'correct': correct,
            'mastery_vector': self.tracker.get_mastery_vector()}

    def _obs(self):
        m = self.tracker.get_mastery_vector() if self.tracker else [0.2] * 5
        return np.array(m + [self.step_count / self.max_steps], dtype=np.float32)

# Quick env smoke test
env = TutoringEnv()
obs, _ = env.reset(seed=42)
print('initial obs:', obs)
for a in [2, 2, 4, 3, 0]:
    obs, r, term, trunc, info = env.step(a)
    print(f'a={a}  diff={info["difficulty"]}  correct={info["correct"]}  reward={r:+.4f}  P(L_w)={info["mastery_vector"]}')

## 5 · Train PPO (Paper Section 5.3.2 hyperparameters)

GPU is auto-detected by SB3 via `device='auto'`. On Colab T4 the full 500K
run finishes in **5–8 minutes**.

TensorBoard logs are written to `logs/ppo_tutor/`; uncomment the
`%load_ext tensorboard` cell at the bottom to visualise training curves
in Colab.

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

TOTAL_TIMESTEPS = 500_000   # Paper default. Use 50_000 for a 1-min smoke run.
OUTPUT_DIR = 'ppo_agent'
os.makedirs(OUTPUT_DIR, exist_ok=True)

env      = make_vec_env(lambda: TutoringEnv(), n_envs=4)
eval_env = make_vec_env(lambda: TutoringEnv(), n_envs=1)

model = PPO(
    policy='MlpPolicy', env=env,
    learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
    verbose=1, tensorboard_log='logs/ppo_tutor/',
    policy_kwargs=dict(net_arch=[64, 64]),
    device='auto',
)

eval_cb = EvalCallback(
    eval_env, best_model_save_path=OUTPUT_DIR, log_path='logs/eval/',
    eval_freq=10_000, n_eval_episodes=20, deterministic=True,
)

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_cb)
final_path = os.path.join(OUTPUT_DIR, 'final_model')
model.save(final_path)
print('✅ saved →', final_path + '.zip')

## 6 · Sanity-check the trained policy

Roll the policy out for a 20-step session and watch the difficulty trajectory
and mastery curve. After 500K steps the policy typically picks moderate
difficulty (3–4) when mastery is mid-range and harder questions when mastery
is high — exactly the shape we want for paper Figure 3.

In [ ]:
obs, _ = env.envs[0].reset(seed=123)
trajectory = []
for t in range(20):
    a, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, info = env.envs[0].step(int(a))
    trajectory.append({'t': t, 'difficulty': info['difficulty'],
                       'correct': info['correct'], 'reward': float(r),
                       'avg_mastery': float(np.mean(info['mastery_vector']))})
    if term: break

import pandas as pd
df = pd.DataFrame(trajectory); df

## 7 · Download the checkpoint

On **Colab** this triggers a browser download of `final_model.zip`.
On a local Jupyter the file is at `ppo_agent/final_model.zip`.

Replace `backend/models/ppo_agent/final_model.zip` in the AImentor repo
with the downloaded file, then restart the FastAPI backend.

In [ ]:
from pathlib import Path
ckpt = Path(OUTPUT_DIR) / 'final_model.zip'
print('size:', ckpt.stat().st_size / 1024, 'KB')

try:
    from google.colab import files  # only on Colab
    files.download(str(ckpt))
except ImportError:
    print('Local Jupyter — file is at:', ckpt.resolve())

## 8 · (Optional) TensorBoard

```python
%load_ext tensorboard
%tensorboard --logdir logs/
```